# Notebook 01b: EIA-930 Historical Backfill (2018–2024)

**One Sensor, One Year — Edition 2: US Grid**

Mirror of `01_download_eia930.ipynb`, parameterized over years. Pulls **US48-only** hourly fuel-mix for each historical year and writes `us48_YYYY_hourly.csv` + `us48_YYYY_daily.csv`. Skips years already on disk — re-runs are free.

**Scope:** 2018-07-01 → 2024-12-31 (EIA-930 v2 starts July 2018). This plus existing `us48_2025_*.csv` gives us 2018–2025 coverage for the cross-edition three-grids plot and for a future US YoY notebook. BA-level data is *not* backfilled here — keeps runtime ~15 min per year vs. hours.

**Outputs:**
- `../data/raw/eia930/US48_{year}.json` — raw API cache per year
- `../data/processed/us48_{year}_hourly.csv` — 8760 rows (≈4380 for 2018)
- `../data/processed/us48_{year}_daily.csv` — ~365 rows

In [1]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

RAW = Path('../data/raw/eia930')
PROCESSED = Path('../data/processed')
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

API_URL = 'https://api.eia.gov/v2/electricity/rto/fuel-type-data/data/'
YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024]
RESPONDENT = 'US48'

load_dotenv(dotenv_path=Path('../../../.env'))
API_KEY = os.getenv('EIA_API_KEY')
assert API_KEY and len(API_KEY) == 32, 'EIA_API_KEY missing or wrong length in .env'
print(f'API key loaded (masked: {API_KEY[:4]}...{API_KEY[-4:]})')
print(f'Target years: {YEARS}')

API key loaded (masked: c09e...0e58)
Target years: [2018, 2019, 2020, 2021, 2022, 2023, 2024]


## 1. Fetch helper (same as 01, year-aware start/end)

Note: EIA-930 v2 hourly data starts **2018-07-01 00:00 UTC**. For 2018 we use that as the `start`. Every other year gets the full Jan 1 → Dec 31 window.

In [2]:
def fetch_eia930_us48(year, sleep_s=0.25, page_size=5000, max_retries=4):
    """Pull all hourly US48 rows for one year, paginating until exhausted. Cached per year."""
    cache = RAW / f'{RESPONDENT}_{year}.json'
    if cache.exists():
        with open(cache) as f:
            data = json.load(f)
        print(f'  [{RESPONDENT} {year}] cached: {len(data):,} rows')
        return data

    start = '2018-07-01T00' if year == 2018 else f'{year}-01-01T00'
    end = f'{year}-12-31T23'
    base_params = {
        'api_key': API_KEY,
        'frequency': 'hourly',
        'data[0]': 'value',
        'facets[respondent][]': RESPONDENT,
        'start': start,
        'end': end,
        'sort[0][column]': 'period',
        'sort[0][direction]': 'asc',
    }

    all_rows, offset, current_page_size = [], 0, page_size
    while True:
        params = {**base_params, 'length': current_page_size, 'offset': offset}
        page = None
        for attempt in range(1, max_retries + 1):
            try:
                r = requests.get(API_URL, params=params, timeout=120)
                if r.status_code >= 500:
                    raise requests.HTTPError(f'{r.status_code} server error')
                r.raise_for_status()
                page = r.json().get('response', {}).get('data', [])
                break
            except (requests.HTTPError, requests.Timeout, requests.ConnectionError) as e:
                if attempt == max_retries:
                    raise RuntimeError(
                        f'  [{RESPONDENT} {year}] failed at offset={offset} after {max_retries} retries: {e}'
                    ) from None
                backoff = 2 ** attempt
                print(f'  [{RESPONDENT} {year}] offset={offset} attempt {attempt} failed ({e}); retrying in {backoff}s')
                time.sleep(backoff)
                if '504' in str(e) and current_page_size > 500:
                    current_page_size = max(500, current_page_size // 2)
                    params['length'] = current_page_size
                    print(f'  [{RESPONDENT} {year}] reduced page size to {current_page_size}')
        if not page:
            break
        all_rows.extend(page)
        offset += current_page_size
        if len(page) < current_page_size:
            break
        time.sleep(sleep_s)

    with open(cache, 'w') as f:
        json.dump(all_rows, f)
    print(f'  [{RESPONDENT} {year}] fetched: {len(all_rows):,} rows -> {cache.name}')
    return all_rows

## 2. Pull all years

Each year is independent and cached. Expect ~10–15 min per year the first time, instant on re-run.

In [3]:
raw_by_year = {}
t0 = time.time()
for year in YEARS:
    raw_by_year[year] = fetch_eia930_us48(year)
elapsed = time.time() - t0
total_rows = sum(len(v) for v in raw_by_year.values())
print(f'\nDone: {total_rows:,} total rows across {len(YEARS)} years in {elapsed:.0f}s')

  [US48 2018] fetched: 0 rows -> US48_2018.json


  [US48 2019] fetched: 70,080 rows -> US48_2019.json


  [US48 2020] fetched: 70,272 rows -> US48_2020.json


  [US48 2021] fetched: 70,080 rows -> US48_2021.json


  [US48 2022] fetched: 70,080 rows -> US48_2022.json


  [US48 2023] fetched: 70,080 rows -> US48_2023.json


  [US48 2024] fetched: 76,076 rows -> US48_2024.json

Done: 426,668 total rows across 7 years in 208s


## 3. Pivot each year to hourly + daily wide CSVs

Same schema as `us48_2025_hourly.csv`: one row per hour, one column per fuel code.

In [4]:
results = {}
for year, rows in raw_by_year.items():
    if not rows:
        print(f'WARN: {year} has no data; skipping')
        continue
    df = pd.DataFrame(rows)
    df['period'] = pd.to_datetime(df['period'], format='%Y-%m-%dT%H')
    df['value_mwh'] = pd.to_numeric(df['value'], errors='coerce')

    hourly = (df.pivot_table(index='period', columns='fueltype', values='value_mwh', aggfunc='sum')
                .sort_index())
    hourly.columns.name = None
    daily = hourly.resample('1D').sum(min_count=1)
    daily.index.name = 'date'
    hourly.index.name = 'period'

    hourly_path = PROCESSED / f'us48_{year}_hourly.csv'
    daily_path = PROCESSED / f'us48_{year}_daily.csv'
    hourly.to_csv(hourly_path)
    daily.to_csv(daily_path)

    results[year] = {
        'hourly_rows': len(hourly),
        'daily_rows': len(daily),
        'fuels': list(hourly.columns),
        'annual_twh': hourly.sum().sum() / 1e6,
        'hourly_path': str(hourly_path),
        'daily_path': str(daily_path),
    }
    print(f'  {year}: {len(hourly):,} hourly rows, {len(daily)} daily rows, {hourly.sum().sum() / 1e6:,.1f} TWh total')

WARN: 2018 has no data; skipping


  2019: 8,760 hourly rows, 365 daily rows, 3,959.1 TWh total


  2020: 8,784 hourly rows, 366 daily rows, 3,815.2 TWh total


  2021: 8,760 hourly rows, 365 daily rows, 3,948.2 TWh total


  2022: 8,760 hourly rows, 365 daily rows, 4,078.4 TWh total
  2023: 8,760 hourly rows, 365 daily rows, 4,039.2 TWh total


  2024: 8,784 hourly rows, 366 daily rows, 4,169.5 TWh total


## 4. Validation — annual totals over time

Rough EIA AEO reference points (approx US net generation, TWh):

| Year | TWh (ref) |
| --- | --- |
| 2018 | ~4,176 (half-year for us, ~2,100 expected) |
| 2019 | ~4,118 |
| 2020 | ~4,009 |
| 2021 | ~4,115 |
| 2022 | ~4,240 |
| 2023 | ~4,176 |
| 2024 | ~4,259 |
| 2025 | ~4,269 (from 01) |

Expect each year within ~2% of reference. 2018 is half-year by design.

In [5]:
summary = pd.DataFrame([
    {'year': y, 'hourly_rows': r['hourly_rows'], 'daily_rows': r['daily_rows'], 'annual_twh': round(r['annual_twh'], 1)}
    for y, r in sorted(results.items())
])
print(summary.to_string(index=False))

 year  hourly_rows  daily_rows  annual_twh
 2019         8760         365      3959.1
 2020         8784         366      3815.2
 2021         8760         365      3948.2
 2022         8760         365      4078.4
 2023         8760         365      4039.2
 2024         8784         366      4169.5


## Next

With `us48_{2018..2025}_daily.csv` now on disk, the cross-edition notebook (`cross-edition/notebooks/01_three_grids_monthly.ipynb`) can load all 8 years at once. US-grid NB 06 (YoY) is unblocked too.